# BBEH × DSPy Optimizers (GEPA, MIPROv2, Bootstrap)

Runs DSPy prompt optimizers on BBEH mini (460 examples, 23 tasks) using `gpt-oss-120b` via Groq.

Toggle which optimizers to run in the config cell below. Each produces an independent `results_*.json` with identical schema.

**Prerequisites:** Add your Groq API key to Colab Secrets as `GROQ_API_KEY`.

In [ ]:
# ══ CONFIG: Toggle which optimizers to run ══
RUN_GEPA = True
RUN_MIPRO = True
RUN_BOOTSTRAP = True

In [ ]:
# Cell: Install dependencies
!pip install -q dspy datasets

In [ ]:
# Cell: Shared config
%%writefile shared_config.py

"""Shared constants for BBEH competitor comparison notebooks."""

MODEL_ID = "gpt-oss-120b"
API_BASE = "https://api.groq.com/openai/v1"
HF_DATASET = "BBEH/bbeh"
SPLIT_SEED = 42
TRAIN_PER_TASK = 10
TEST_PER_TASK = 10


def exact_match(expected: str, predicted: str) -> bool:
    return expected.strip().lower() == predicted.strip().lower()


def load_and_split():
    import random
    from datasets import load_dataset

    ds = load_dataset(HF_DATASET)["train"]
    mini = ds.filter(lambda x: x["mini"] == 1)

    by_task: dict[str, list[dict]] = {}
    for ex in mini:
        task = ex["task"]
        by_task.setdefault(task, []).append({"input": ex["input"], "target": ex["target"]})

    train_by_task: dict[str, list[dict]] = {}
    test_by_task: dict[str, list[dict]] = {}

    for task, examples in sorted(by_task.items()):
        rng = random.Random(SPLIT_SEED)
        shuffled = examples.copy()
        rng.shuffle(shuffled)
        train_by_task[task] = shuffled[:TRAIN_PER_TASK]
        test_by_task[task] = shuffled[TRAIN_PER_TASK:TRAIN_PER_TASK + TEST_PER_TASK]

    return train_by_task, test_by_task


def export_results(method, per_task, config, optimized_prompts, output_path="results.json"):
    import json
    from datetime import datetime, timezone

    accuracies = [t["accuracy"] for t in per_task.values()]
    overall = sum(accuracies) / len(accuracies) if accuracies else 0.0

    result = {
        "method": method,
        "model": MODEL_ID,
        "dataset": "bbeh-mini",
        "split_seed": SPLIT_SEED,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "config": config,
        "per_task": per_task,
        "overall_accuracy": round(overall, 4),
        "optimized_prompts": optimized_prompts,
    }

    with open(output_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"Results written to {output_path}")
    print(f"Overall accuracy (macro-avg): {overall:.1%}")
    return result

In [ ]:
# Cell: Configure DSPy with Groq + load dataset
import os
import dspy
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

from shared_config import (
    MODEL_ID, API_BASE, load_and_split,
    exact_match, export_results,
)

# Inference LM (high volume, deterministic)
lm = dspy.LM(
    f"openai/{MODEL_ID}",
    api_key=GROQ_API_KEY,
    api_base=API_BASE,
    max_tokens=512,
    temperature=0.0,
)

# Reflection / proposal LM (same model, higher temperature for diversity)
reflection_lm = dspy.LM(
    f"openai/{MODEL_ID}",
    api_key=GROQ_API_KEY,
    api_base=API_BASE,
    max_tokens=2048,
    temperature=1.0,
)

dspy.configure(lm=lm)

train_by_task, test_by_task = load_and_split()
tasks = sorted(train_by_task.keys())
print(f"Configured DSPy with {MODEL_ID} via {API_BASE}")
print(f"Loaded {len(tasks)} tasks, {sum(len(v) for v in train_by_task.values())} train, "
      f"{sum(len(v) for v in test_by_task.values())} test")

In [ ]:
# Cell: Define DSPy program, metric, and helpers

class BBEHTask(dspy.Signature):
    """Solve the given reasoning problem. Provide only the final answer."""
    problem = dspy.InputField(desc="The reasoning problem to solve")
    answer = dspy.OutputField(desc="The final answer")


def bbeh_metric(gold, pred, trace=None):
    """Exact-match metric with diagnostic feedback (required by GEPA)."""
    predicted = (pred.answer or "").strip()
    expected = gold.answer.strip()
    is_correct = exact_match(expected, predicted)
    score = 1.0 if is_correct else 0.0

    if is_correct:
        feedback = f"Correct. The answer is '{expected}'."
    else:
        feedback = (
            f"Incorrect. Expected '{expected}', got '{predicted}'. "
            f"Re-examine the reasoning steps and check for errors."
        )

    return dspy.Prediction(score=score, feedback=feedback)


def to_dspy_examples(examples: list[dict]) -> list[dspy.Example]:
    """Convert shared-format dicts to DSPy Examples."""
    return [
        dspy.Example(
            problem=ex["input"],
            answer=ex["target"],
        ).with_inputs("problem")
        for ex in examples
    ]


def run_optimizer(name, make_optimizer, tasks, train_by_task):
    """Run an optimizer across all 23 tasks. Returns (programs, prompts)."""
    programs = {}
    prompts = {}

    for i, task in enumerate(tasks):
        print(f"\n[{name}] [{i+1}/{len(tasks)}] Optimizing: {task}")
        trainset = to_dspy_examples(train_by_task[task])
        program = dspy.ChainOfThought(BBEHTask)
        optimizer = make_optimizer()

        try:
            optimized = optimizer.compile(
                student=program,
                trainset=trainset,
                valset=trainset,  # reuse train as val (only 10 examples)
            )
            programs[task] = optimized
            prompts[task] = str(optimized)
            print(f"  Done: {prompts[task][:80]}...")
        except Exception as e:
            print(f"  FAILED: {e}")
            programs[task] = dspy.ChainOfThought(BBEHTask)
            prompts[task] = f"FAILED: {e}"

    return programs, prompts


def evaluate_and_export(name, programs, test_by_task, config):
    """Evaluate programs on held-out test set, print summary, export JSON."""
    per_task = {}
    prompts = {task: str(prog) for task, prog in programs.items()}

    for i, task in enumerate(sorted(programs)):
        program = programs[task]
        test_examples = to_dspy_examples(test_by_task[task])
        correct = 0

        for ex in test_examples:
            try:
                pred = program(problem=ex.problem)
                if exact_match(ex.answer, pred.answer or ""):
                    correct += 1
            except Exception as e:
                print(f"  [{name}] Error on {task}: {e}")

        accuracy = correct / len(test_examples)
        per_task[task] = {"accuracy": round(accuracy, 4), "n_test": len(test_examples)}

    # Print summary
    print(f"\n{'='*50}")
    print(f"{name.upper()} RESULTS")
    print(f"{'='*50}")
    print(f"{'Task':<25} {'Accuracy':>8}")
    print(f"{'-'*50}")
    for task in sorted(per_task):
        acc = per_task[task]["accuracy"]
        print(f"{task:<25} {acc:>8.0%}")

    output_path = f"results_{name}.json"
    result = export_results(name, per_task, config, prompts, output_path)
    return result


# Collect all results for final comparison
all_results = {}

---
## GEPA
Reflective prompt evolution with error-driven feedback. [arXiv:2507.19457](https://arxiv.org/abs/2507.19457) (ICLR 2026 Oral)

In [ ]:
if RUN_GEPA:
    gepa_config = {"optimizer": "gepa", "auto": "light", "num_threads": 4}

    def make_gepa():
        return dspy.GEPA(
            metric=bbeh_metric,
            auto="light",
            reflection_lm=reflection_lm,
            num_threads=4,
            track_stats=True,
        )

    gepa_programs, gepa_prompts = run_optimizer("gepa", make_gepa, tasks, train_by_task)
    all_results["gepa"] = evaluate_and_export("gepa", gepa_programs, test_by_task, gepa_config)
else:
    print("GEPA skipped (RUN_GEPA = False)")

---
## MIPROv2
Bayesian optimization over instructions + few-shot demos. [arXiv:2406.11695](https://arxiv.org/abs/2406.11695) (EMNLP 2024)

In [ ]:
if RUN_MIPRO:
    from dspy.teleprompt import MIPROv2

    mipro_config = {"optimizer": "miprov2", "auto": "light", "num_threads": 4}

    def make_mipro():
        return MIPROv2(
            metric=bbeh_metric,
            auto="light",
            num_threads=4,
            verbose=True,
        )

    mipro_programs, mipro_prompts = run_optimizer("miprov2", make_mipro, tasks, train_by_task)
    all_results["miprov2"] = evaluate_and_export("miprov2", mipro_programs, test_by_task, mipro_config)
else:
    print("MIPROv2 skipped (RUN_MIPRO = False)")

---
## BootstrapFewShotWithRandomSearch
Random search over bootstrapped few-shot demonstrations. [DSPy docs](https://dspy.ai/api/optimizers/BootstrapFewShotWithRandomSearch/)

In [ ]:
if RUN_BOOTSTRAP:
    from dspy.teleprompt import BootstrapFewShotWithRandomSearch

    bootstrap_config = {
        "optimizer": "bootstrap_fewshot",
        "max_bootstrapped_demos": 4,
        "max_labeled_demos": 4,
        "num_candidate_programs": 8,
        "num_threads": 4,
    }

    def make_bootstrap():
        return BootstrapFewShotWithRandomSearch(
            metric=bbeh_metric,
            max_bootstrapped_demos=4,
            max_labeled_demos=4,
            num_candidate_programs=8,
            num_threads=4,
        )

    bootstrap_programs, bootstrap_prompts = run_optimizer(
        "bootstrap_fewshot", make_bootstrap, tasks, train_by_task
    )
    all_results["bootstrap_fewshot"] = evaluate_and_export(
        "bootstrap_fewshot", bootstrap_programs, test_by_task, bootstrap_config
    )
else:
    print("Bootstrap skipped (RUN_BOOTSTRAP = False)")

---
## Comparison

In [ ]:
# Side-by-side comparison of all optimizers that ran
if len(all_results) < 2:
    print("Only one optimizer ran — nothing to compare.")
else:
    methods = sorted(all_results.keys())
    header = f"{'Task':<25}" + "".join(f"{m:>16}" for m in methods)
    print(header)
    print("-" * len(header))

    for task in tasks:
        row = f"{task:<25}"
        for m in methods:
            acc = all_results[m]["per_task"].get(task, {}).get("accuracy", -1)
            row += f"{acc:>15.0%} " if acc >= 0 else f"{'N/A':>16}"
        print(row)

    print("-" * len(header))
    row = f"{'MACRO AVERAGE':<25}"
    for m in methods:
        row += f"{all_results[m]['overall_accuracy']:>15.1%} "
    print(row)